In [ ]:
# CELL 1 - Runtime Information
from pathlib import Path
import sys, platform
from importlib.metadata import version
print("Python:",sys.version); print("Platform:",platform.platform()); print("Working directory:",Path.cwd())
for p in ["pandas","numpy","matplotlib","pillow","pytest"]: print(p,version(p))


In [ ]:
# CELL 2 - Install Dependencies
import subprocess,sys
from pathlib import Path
Path("/content/requirements_stage7.txt").write_text('pandas>=2.2,<3.0\nnumpy>=1.26,<3.0\nmatplotlib>=3.8,<4.0\npillow>=10.0,<13.0\npytest>=8.0,<10.0\n',encoding="utf-8")
subprocess.run([sys.executable,"-m","pip","install","-q","-r","/content/requirements_stage7.txt"],check=True)


In [ ]:
# CELL 3 - Create Project Directories
PROJECT_ROOT=Path("/content/DT25_Stage7_EDA_Execution")
PATHS={"src":PROJECT_ROOT/"src","tests":PROJECT_ROOT/"tests","input":PROJECT_ROOT/"data"/"input","tables":PROJECT_ROOT/"outputs"/"tables"/"analysis","figures":PROJECT_ROOT/"outputs"/"figures"/"analysis","logs":PROJECT_ROOT/"outputs"/"logs"/"stage7"}
for p in PATHS.values(): p.mkdir(parents=True,exist_ok=True)
print(PROJECT_ROOT)


In [ ]:
# CELL 4 - Upload Required Inputs
from google.colab import files
import shutil
required={"online_retail_clean.csv","online_retail_rfm_eligible.csv","online_retail_cancellations_returns.csv","rfm_customers.csv","rfm_with_scores.csv"}
uploaded=files.upload()
if set(uploaded)!=required: raise ValueError(f"Upload exactly {sorted(required)}; received {sorted(uploaded)}")
for name in required:
    source=Path("/content")/name
    if not source.exists() or source.stat().st_size<=0: raise ValueError(f"Missing/empty upload: {name}")
    shutil.copy2(source,PATHS["input"]/name)
print("Uploaded:",sorted(required))


In [ ]:
# CELL 5 - Validate Input Files
# Full validation occurs in AnalysisEngine.validate_inputs after module import.
for p in sorted(PATHS["input"].glob("*.csv")): print(p.name,p.stat().st_size)


In [ ]:
# CELL 6 - Input SHA-256 Baseline
import hashlib
def sha256(p):
 h=hashlib.sha256()
 with p.open("rb") as f:
  for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
 return h.hexdigest()
pre_hash={p.name:sha256(p) for p in PATHS["input"].glob("*.csv")}
print(pre_hash)


In [ ]:
# CELL 7 - Write and Import analysis_engine.py
import sys,importlib
module_source='"""Stage 7 EDA engine for the seven locked DT25 analysis questions.\n\nThe engine consumes accepted Stage 4/6 CSV outputs, creates analysis tables,\nfigures, and traceability registries. It does not run clustering, choose k,\ncreate PCA, or assign final customer segments.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\nimport hashlib\nimport json\nimport re\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom PIL import Image\n\nEXPECTED = {\n    "online_retail_clean.csv": {"rows": 524878, "customers": None},\n    "online_retail_rfm_eligible.csv": {"rows": 392692, "customers": 4338},\n    "online_retail_cancellations_returns.csv": {"rows": 11763, "customers": None},\n    "rfm_customers.csv": {"rows": 4338, "customers": 4338},\n    "rfm_with_scores.csv": {"rows": 4338, "customers": 4338},\n}\nQUESTIONS = ["Q01","Q02","Q03","Q04","Q05","Q06","Q07"]\nREQUIRED_TRANSACTION = ["InvoiceNo","StockCode","Description","Quantity","InvoiceDate","UnitPrice","CustomerID","Country","TransactionAmount"]\n\nclass AnalysisInputError(ValueError): pass\nclass AnalysisAcceptanceError(RuntimeError): pass\n\n@dataclass(frozen=True)\nclass Paths:\n    root: Path\n    @property\n    def input(self): return self.root/"data"/"input"\n    @property\n    def tables(self): return self.root/"outputs"/"tables"/"analysis"\n    @property\n    def figures(self): return self.root/"outputs"/"figures"/"analysis"\n    @property\n    def logs(self): return self.root/"outputs"/"logs"/"stage7"\n    def create(self):\n        for p in (self.tables,self.figures,self.logs): p.mkdir(parents=True,exist_ok=True)\n\nclass AnalysisEngine:\n    """Execute the approved EDA registry with code-derived interpretations."""\n    def __init__(self, paths: Paths):\n        self.paths=paths; self.data={}; self.hash_before={}; self.hash_after={}\n        self.results=[]; self.figures=[]; self.interpretations=[]; self.limitations=[]\n\n    @staticmethod\n    def sha256(path: Path)->str:\n        h=hashlib.sha256()\n        with path.open("rb") as f:\n            for b in iter(lambda:f.read(1024*1024),b""): h.update(b)\n        return h.hexdigest()\n\n    def validate_inputs(self)->pd.DataFrame:\n        self.paths.create(); rows=[]\n        schemas={\n            "online_retail_clean.csv": REQUIRED_TRANSACTION,\n            "online_retail_rfm_eligible.csv": REQUIRED_TRANSACTION,\n            "online_retail_cancellations_returns.csv": ["InvoiceNo","StockCode","InvoiceDate","Quantity","UnitPrice","Country","IsCancellation","IsNegativeQuantity","IsInvalidUnitPrice"],\n            "rfm_customers.csv": ["CustomerID","LastPurchaseDate","Recency","Frequency","Monetary"],\n            "rfm_with_scores.csv": ["CustomerID","Recency","Frequency","Monetary","R_Score","F_Score","M_Score","RFM_TotalScore"],\n        }\n        for name,baseline in EXPECTED.items():\n            p=self.paths.input/name\n            if not p.exists() or p.stat().st_size<=0: raise FileNotFoundError(f"Missing/empty input: {name}")\n            self.hash_before[name]=self.sha256(p)\n            try: df=pd.read_csv(p,dtype={"CustomerID":"string","InvoiceNo":"string"},low_memory=False)\n            except (OSError,pd.errors.ParserError,UnicodeDecodeError) as e: raise AnalysisInputError(f"Cannot read {name}: {e}") from e\n            missing=sorted(set(schemas[name])-set(df.columns))\n            if missing: raise AnalysisInputError(f"{name} missing columns: {missing}")\n            if len(df)!=baseline["rows"]: raise AnalysisInputError(f"{name} rows={len(df)} expected={baseline[\'rows\']}")\n            if baseline["customers"] is not None and df.CustomerID.nunique()!=baseline["customers"]: raise AnalysisInputError(f"{name} customers mismatch")\n            if "InvoiceDate" in df: df["InvoiceDate"]=pd.to_datetime(df.InvoiceDate,errors="coerce")\n            for c in ["Quantity","UnitPrice","TransactionAmount","Recency","Frequency","Monetary"]:\n                if c in df: df[c]=pd.to_numeric(df[c],errors="coerce")\n            self.data[name]=df\n            rows.append({"File":name,"Rows":len(df),"Columns":df.shape[1],"Customers":int(df.CustomerID.nunique()) if "CustomerID" in df else None,"MissingCritical":int(df[schemas[name]].isna().sum().sum()),"SizeBytes":p.stat().st_size,"SHA256":self.hash_before[name],"Status":"PASS"})\n        if self.data["rfm_customers.csv"].CustomerID.duplicated().any(): raise AnalysisInputError("Duplicate CustomerID in rfm_customers.csv")\n        return pd.DataFrame(rows)\n\n    def _save_table(self,qid,df,name):\n        if df.empty: raise AnalysisAcceptanceError(f"Empty table for {qid}")\n        p=self.paths.tables/name; df.to_csv(p,index=False,encoding="utf-8-sig")\n        return p\n    def _save_fig(self,qid,fig,figure_id,chart_type,title,xlabel,ylabel,unit,name,source,caption):\n        if not title or not xlabel or not ylabel: raise AnalysisAcceptanceError(f"Missing chart metadata {figure_id}")\n        p=self.paths.figures/name; fig.tight_layout(); fig.savefig(p,dpi=160,bbox_inches="tight"); plt.close(fig)\n        self.figures.append({"FigureID":figure_id,"QuestionID":qid,"ChartType":chart_type,"Title":title,"XLabel":xlabel,"YLabel":ylabel,"Unit":unit,"Path":str(p.relative_to(self.paths.root)),"SourceDataset":source,"Caption":caption})\n        return p\n    def _record(self,qid,question,dataset,table,figure,cell,points,limits,owner,checker,rubric):\n        self.results.append({"QuestionID":qid,"Question":question,"Dataset":dataset,"TablePath":str(table.relative_to(self.paths.root)),"FigurePath":str(figure.relative_to(self.paths.root)),"CellID":cell,"Owner":owner,"CrossChecker":checker,"RubricMapping":rubric,"TechnicalStatus":"TECHNICALLY PASS, PEER REVIEW PENDING","PeerReviewStatus":"PENDING"})\n        for i,text in enumerate(points,1): self.interpretations.append({"QuestionID":qid,"PointID":f"{qid}-I{i}","Text":text,"SourceTable":str(table.relative_to(self.paths.root)),"ClaimType":"DESCRIPTIVE"})\n        for i,text in enumerate(limits,1): self.limitations.append({"QuestionID":qid,"LimitationID":f"{qid}-L{i}","Text":text})\n\n    def analyze_time_trend(self):\n        q="Số hóa đơn, số giao dịch và tổng TransactionAmount của các giao dịch mua hợp lệ thay đổi theo tháng như thế nào?"; df=self.data["online_retail_clean.csv"]\n        t=df.assign(Month=df.InvoiceDate.dt.to_period("M").astype(str)).groupby("Month",as_index=False).agg(LineRows=("InvoiceNo","size"),UniqueInvoices=("InvoiceNo","nunique"),UniqueCustomers=("CustomerID","nunique"),Revenue=("TransactionAmount","sum"))\n        table=self._save_table("Q01",t,"q01_monthly_trends.csv")\n        fig,ax=plt.subplots(figsize=(11,5)); ax.plot(t.Month,t.Revenue,marker="o"); ax.set_title("FIG-Q01-01: Xu hướng doanh thu mua hàng theo tháng"); ax.set_xlabel("Tháng giao dịch"); ax.set_ylabel("Doanh thu (sterling)"); ax.tick_params(axis="x",rotation=45)\n        figure=self._save_fig("Q01",fig,"FIG-Q01-01","Line chart",ax.get_title(),"Tháng giao dịch","Doanh thu","sterling","fig_q01_monthly_revenue.png","online_retail_clean.csv","Dữ liệu theo tháng; tháng biên có thể không đầy đủ.")\n        peak=t.loc[t.Revenue.idxmax()]; low=t.loc[t.Revenue.idxmin()]\n        pts=[f"Tháng có doanh thu tổng cao nhất trong bảng là {peak.Month}, giá trị {peak.Revenue:.2f} sterling.",f"Tháng có doanh thu tổng thấp nhất trong bảng là {low.Month}, giá trị {low.Revenue:.2f} sterling.",f"Khoảng quan sát chạy từ {t.Month.iloc[0]} đến {t.Month.iloc[-1]}; các tháng biên phải được diễn giải như giai đoạn quan sát không nhất thiết đầy đủ."]\n        self._record("Q01",q,"online_retail_clean.csv",table,figure,"S7-Q01",pts,["Chuỗi quan sát ngắn không đủ để khẳng định quy luật mùa vụ dài hạn.","Biến động mô tả không chứng minh nguyên nhân."],"TV3","TV1","Xu hướng thời gian; groupby; doanh thu; line chart")\n\n    def analyze_revenue_contribution(self):\n        q="Doanh thu mua hàng, số hóa đơn và số khách hàng có định danh phân bố theo quốc gia như thế nào, và mức độ tập trung đóng góp ra sao?"; df=self.data["online_retail_clean.csv"]\n        t=df.groupby("Country",as_index=False).agg(Revenue=("TransactionAmount","sum"),UniqueInvoices=("InvoiceNo","nunique"),UniqueCustomers=("CustomerID","nunique"),Lines=("InvoiceNo","size")).sort_values("Revenue",ascending=False); t["RevenueSharePercent"]=t.Revenue/t.Revenue.sum()*100; t["CumulativeSharePercent"]=t.RevenueSharePercent.cumsum()\n        table=self._save_table("Q02",t,"q02_country_contribution.csv"); top=t.head(15).sort_values("Revenue")\n        fig,ax=plt.subplots(figsize=(10,7)); ax.barh(top.Country,top.Revenue); ax.set_title("FIG-Q02-01: Đóng góp doanh thu theo quốc gia"); ax.set_xlabel("Doanh thu (sterling)"); ax.set_ylabel("Quốc gia")\n        figure=self._save_fig("Q02",fig,"FIG-Q02-01","Horizontal bar chart",ax.get_title(),"Doanh thu","Quốc gia","sterling","fig_q02_country_revenue.png","online_retail_clean.csv","15 quốc gia có doanh thu cao nhất; bảng CSV giữ toàn bộ quốc gia.")\n        first=t.iloc[0]\n        pts=[f"Quốc gia đứng đầu bảng doanh thu là {first.Country}, đạt {first.Revenue:.2f} sterling và chiếm {first.RevenueSharePercent:.2f}% tổng doanh thu mua hợp lệ.",f"Bảng đầy đủ gồm {len(t)} quốc gia và sử dụng số hóa đơn duy nhất thay vì số dòng sản phẩm.",f"Số khách hàng theo quốc gia chỉ đếm CustomerID không thiếu."]\n        self._record("Q02",q,"online_retail_clean.csv",table,figure,"S7-Q02",pts,["Doanh thu không phải lợi nhuận và không cung cấp chi phí hoặc ROI.","Country không chứng minh tiềm năng thị trường hay nguyên nhân doanh thu."],"TV3","TV2","So sánh nhóm; doanh thu; groupby; horizontal bar")\n\n    def analyze_customer_behavior(self):\n        q="Giá trị hóa đơn, số loại sản phẩm trong hóa đơn và số hóa đơn theo khách hàng phân bố như thế nào?"; df=self.data["online_retail_rfm_eligible.csv"]\n        inv=df.groupby(["CustomerID","InvoiceNo"],as_index=False).agg(InvoiceValue=("TransactionAmount","sum"),UniqueProducts=("StockCode","nunique"),TotalQuantity=("Quantity","sum")); cust=df.groupby("CustomerID",as_index=False).agg(CustomerInvoices=("InvoiceNo","nunique"))\n        metrics=pd.DataFrame([{"Metric":"InvoiceValue","Count":len(inv),"Mean":inv.InvoiceValue.mean(),"Median":inv.InvoiceValue.median(),"Q1":inv.InvoiceValue.quantile(.25),"Q3":inv.InvoiceValue.quantile(.75),"Max":inv.InvoiceValue.max()},{"Metric":"UniqueProducts","Count":len(inv),"Mean":inv.UniqueProducts.mean(),"Median":inv.UniqueProducts.median(),"Q1":inv.UniqueProducts.quantile(.25),"Q3":inv.UniqueProducts.quantile(.75),"Max":inv.UniqueProducts.max()},{"Metric":"CustomerInvoices","Count":len(cust),"Mean":cust.CustomerInvoices.mean(),"Median":cust.CustomerInvoices.median(),"Q1":cust.CustomerInvoices.quantile(.25),"Q3":cust.CustomerInvoices.quantile(.75),"Max":cust.CustomerInvoices.max()}])\n        table=self._save_table("Q03",metrics,"q03_customer_behavior.csv")\n        fig,axes=plt.subplots(1,2,figsize=(12,5)); axes[0].hist(inv.InvoiceValue,bins=60); axes[0].set_title("Histogram giá trị hóa đơn"); axes[0].set_xlabel("Giá trị hóa đơn (sterling)"); axes[0].set_ylabel("Số hóa đơn"); axes[1].boxplot([inv.InvoiceValue,cust.CustomerInvoices],labels=["InvoiceValue","Invoices/customer"]); axes[1].set_title("Boxplot hành vi mua"); axes[1].set_xlabel("Chỉ tiêu"); axes[1].set_ylabel("Giá trị")\n        figure=self._save_fig("Q03",fig,"FIG-Q03-01","Histogram + boxplot", "FIG-Q03-01: Phân bố hành vi mua cấp hóa đơn và khách hàng","Chỉ tiêu hành vi","Tần số hoặc giá trị","sterling/count","fig_q03_customer_behavior.png","online_retail_rfm_eligible.csv","Histogram và boxplot dùng toàn bộ dữ liệu hợp lệ; điểm ngoài whisker không tự động là lỗi.")\n        mi=metrics.set_index("Metric")\n        pts=[f"Median giá trị hóa đơn là {mi.loc[\'InvoiceValue\',\'Median\']:.2f} sterling, trong khi mean là {mi.loc[\'InvoiceValue\',\'Mean\']:.2f} sterling.",f"Median số sản phẩm duy nhất trong một hóa đơn là {mi.loc[\'UniqueProducts\',\'Median\']:.0f}.",f"Median số hóa đơn duy nhất theo khách hàng là {mi.loc[\'CustomerInvoices\',\'Median\']:.0f}."]\n        self._record("Q03",q,"online_retail_rfm_eligible.csv",table,figure,"S7-Q03",pts,["Outlier được giữ nguyên và không nên tự động coi là lỗi.","Dữ liệu không chứa phiên truy cập hoặc ý định mua."],"TV3","TV4","Thống kê mô tả; histogram; boxplot; hành vi khách hàng")\n\n    def analyze_cancellations_returns(self):\n        q="Các giao dịch hủy, trả hàng và giao dịch không hợp lệ được tách riêng phân bố như thế nào theo loại lý do, thời gian, quốc gia và mã sản phẩm?"; df=self.data["online_retail_cancellations_returns.csv"]\n        cancel=df.IsCancellation.astype(str).str.lower().eq("true"); neg=df.IsNegativeQuantity.astype(str).str.lower().eq("true"); invp=df.IsInvalidUnitPrice.astype(str).str.lower().eq("true")\n        categories=np.select([cancel&neg,cancel&~neg,~cancel&neg,~cancel&~neg&invp],["Cancellation + negative quantity","Cancellation without negative quantity","Negative quantity without cancellation","Invalid price only"],default="Other exception")\n        x=df.assign(ExceptionCategory=categories,Month=df.InvoiceDate.dt.to_period("M").astype(str)); t=x.groupby(["Month","ExceptionCategory"],as_index=False).size().rename(columns={"size":"Rows"})\n        table=self._save_table("Q04",t,"q04_exception_profile.csv"); p=t.pivot(index="Month",columns="ExceptionCategory",values="Rows").fillna(0)\n        fig,ax=plt.subplots(figsize=(12,6)); p.plot(kind="bar",stacked=True,ax=ax); ax.set_title("FIG-Q04-01: Cơ cấu hủy, trả hàng và giao dịch không hợp lệ theo tháng"); ax.set_xlabel("Tháng"); ax.set_ylabel("Số dòng exception"); ax.legend(title="Nhóm độc quyền",bbox_to_anchor=(1.02,1))\n        figure=self._save_fig("Q04",fig,"FIG-Q04-01","Stacked bar chart",ax.get_title(),"Tháng","Số dòng exception","dòng","fig_q04_exception_composition.png","online_retail_cancellations_returns.csv","Các nhóm loại trừ lẫn nhau, không cộng trực tiếp cancellation và negative quantity.")\n        totals=x.ExceptionCategory.value_counts(); pts=[f"Bảng phân loại {len(x)} dòng exception thành các nhóm độc quyền để tránh cộng chồng lấp.",f"Nhóm lớn nhất trong phép phân loại là {totals.index[0]} với {int(totals.iloc[0])} dòng.",f"Khoảng quan sát exception từ {x.Month.min()} đến {x.Month.max()}."]\n        self._record("Q04",q,"online_retail_cancellations_returns.csv",table,figure,"S7-Q04",pts,["Exception không đồng nghĩa toàn bộ là đơn bị hủy.","Dòng exception không cho biết nguyên nhân vận hành đầy đủ."],"TV2","TV1","Data quality; crosstab/groupby; stacked bar; so sánh nhóm")\n\n    def analyze_rfm(self):\n        q="Recency, Frequency và Monetary phân bố, lệch và liên hệ với nhau như thế nào sau khi bảng RFM được tạo bằng giao dịch đủ điều kiện?"; r=self.data["rfm_customers.csv"]; vars=["Recency","Frequency","Monetary"]\n        d=r[vars].describe(percentiles=[.25,.5,.75,.9,.95,.99]).T.reset_index(names="Variable"); d["Skewness"]=r[vars].skew().values; corr=r[vars].corr(method="spearman"); corr_long=corr.stack().reset_index(); corr_long.columns=["Variable1","Variable2","SpearmanCorrelation"]; table_data=d.merge(pd.DataFrame({"JoinKey":[1]*len(d)}),left_index=True,right_index=True).drop(columns="JoinKey")\n        table=self._save_table("Q05",table_data,"q05_rfm_diagnostics.csv"); corr_long.to_csv(self.paths.tables/"q05_rfm_correlation.csv",index=False,encoding="utf-8-sig")\n        fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(corr,vmin=-1,vmax=1,cmap="coolwarm"); ax.set_xticks(range(3),vars); ax.set_yticks(range(3),vars); ax.set_title("FIG-Q05-01: Tương quan Spearman giữa các chỉ số RFM"); ax.set_xlabel("Chỉ số RFM"); ax.set_ylabel("Chỉ số RFM"); fig.colorbar(im,ax=ax,label="Hệ số Spearman")\n        for i in range(3):\n            for j in range(3): ax.text(j,i,f"{corr.iloc[i,j]:.2f}",ha="center",va="center")\n        figure=self._save_fig("Q05",fig,"FIG-Q05-01","Heatmap",ax.get_title(),"Chỉ số RFM","Chỉ số RFM","hệ số [-1,1]","fig_q05_rfm_correlation.png","rfm_customers.csv","Tương quan Spearman mô tả quan hệ đơn điệu, không chứng minh nhân quả.")\n        skew=d.set_index("Variable").Skewness; off=corr.where(~np.eye(3,dtype=bool)).stack(); pair=off.abs().idxmax(); val=corr.loc[pair]\n        pts=[f"Skewness của Recency, Frequency và Monetary lần lượt là {skew.Recency:.3f}, {skew.Frequency:.3f}, {skew.Monetary:.3f}.",f"Cặp có độ lớn tương quan Spearman ngoài đường chéo cao nhất là {pair[0]} và {pair[1]}, hệ số {val:.3f}.",f"Bảng sử dụng {len(r)} khách hàng và không loại RFM outlier."]\n        self._record("Q05",q,"rfm_customers.csv",table,figure,"S7-Q05",pts,["Tương quan không chứng minh quan hệ nhân quả.","Phân phối lệch không tự động biện minh cho việc loại khách hàng."],"TV3","TV4","RFM; correlation; heatmap; thống kê mô tả")\n\n    def analyze_monetary_concentration(self):\n        q="Monetary có tập trung vào một nhóm nhỏ khách hàng hay không, và nhóm đóng góp Monetary cao có đặc điểm Recency và Frequency như thế nào?"; r=self.data["rfm_customers.csv"].sort_values("Monetary",ascending=False).reset_index(drop=True); r["Rank"]=np.arange(1,len(r)+1); r["MonetarySharePercent"]=r.Monetary/r.Monetary.sum()*100; r["CumulativeSharePercent"]=r.MonetarySharePercent.cumsum(); r["CustomerPercent"]=r.Rank/len(r)*100\n        table=self._save_table("Q06",r[["CustomerID","Rank","Recency","Frequency","Monetary","MonetarySharePercent","CumulativeSharePercent","CustomerPercent"]],"q06_monetary_concentration.csv")\n        fig,ax1=plt.subplots(figsize=(11,6)); n=min(50,len(r)); ax1.bar(r.Rank.head(n),r.Monetary.head(n)); ax1.set_xlabel("Xếp hạng khách hàng theo Monetary"); ax1.set_ylabel("Monetary (sterling)"); ax2=ax1.twinx(); ax2.plot(r.Rank,r.CumulativeSharePercent,color="red"); ax2.set_ylabel("Tỷ trọng tích lũy (%)"); ax1.set_title("FIG-Q06-01: Mức độ tập trung Monetary theo khách hàng")\n        figure=self._save_fig("Q06",fig,"FIG-Q06-01","Pareto chart",ax1.get_title(),"Xếp hạng khách hàng","Monetary và tỷ trọng tích lũy","sterling/%","fig_q06_customer_monetary_pareto.png","rfm_customers.csv","Bar hiển thị 50 khách hàng đầu; đường cumulative dùng toàn bộ khách hàng.")\n        top10=max(1,int(np.ceil(len(r)*.1))); share=r.head(top10).Monetary.sum()/r.Monetary.sum()*100; top=r.iloc[0]\n        pts=[f"Top 10% khách hàng theo Monetary đóng góp {share:.2f}% tổng Monetary trong bảng RFM.",f"Khách hàng đứng đầu có Monetary {top.Monetary:.2f} sterling, Frequency {int(top.Frequency)} và Recency {int(top.Recency)} ngày.",f"Đường cumulative được tính trên toàn bộ {len(r)} khách hàng."]\n        self._record("Q06",q,"rfm_customers.csv",table,figure,"S7-Q06",pts,["Monetary không phải lợi nhuận hoặc customer lifetime value.","Không được gọi quy luật 80/20 nếu bảng không hỗ trợ đúng ngưỡng đó."],"TV3","TV4","Doanh thu; RFM; Pareto; concentration")\n\n    def analyze_descriptive_profile_readiness(self):\n        old="Sau khi mô hình và số cụm được chọn bằng bằng chứng, các cụm khác nhau thế nào về Recency, Frequency, Monetary, quy mô và tỷ trọng Monetary, và hành động chăm sóc nào phù hợp với profile quan sát được?"\n        q="Trước khi clustering, các tổ hợp điểm RFM mô tả phân bố thế nào về quy mô và Monetary, và chúng cung cấp baseline profiling nào để đối chiếu với cụm ở Giai đoạn 9?"\n        s=self.data["rfm_with_scores.csv"]; t=s.groupby(["R_Score","F_Score","M_Score"],as_index=False).agg(Customers=("CustomerID","nunique"),MedianRecency=("Recency","median"),MedianFrequency=("Frequency","median"),MedianMonetary=("Monetary","median"),TotalMonetary=("Monetary","sum")); t["MonetarySharePercent"]=t.TotalMonetary/t.TotalMonetary.sum()*100\n        table=self._save_table("Q07",t,"q07_rfm_score_profile_baseline.csv"); heat=s.pivot_table(index="R_Score",columns="F_Score",values="Monetary",aggfunc="median")\n        fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(heat.values,origin="lower",aspect="auto",cmap="viridis"); ax.set_xticks(range(len(heat.columns)),heat.columns); ax.set_yticks(range(len(heat.index)),heat.index); ax.set_title("FIG-Q07-01: Baseline Monetary theo điểm R và F"); ax.set_xlabel("F_Score"); ax.set_ylabel("R_Score"); fig.colorbar(im,ax=ax,label="Median Monetary (sterling)")\n        figure=self._save_fig("Q07",fig,"FIG-Q07-01","Profile heatmap",ax.get_title(),"F_Score","R_Score","median Monetary (sterling)","fig_q07_rfm_score_profile_baseline.png","rfm_with_scores.csv","Baseline mô tả theo điểm quintile; không phải Cluster ID hoặc phân khúc cuối.")\n        largest=t.loc[t.Customers.idxmax()]; pts=[f"Tổ hợp điểm có nhiều khách hàng nhất trong bảng là R={int(largest.R_Score)}, F={int(largest.F_Score)}, M={int(largest.M_Score)} với {int(largest.Customers)} khách hàng.",f"Heatmap dùng median Monetary để giảm ảnh hưởng của giá trị cực đoan trong phần mô tả.","Kết quả này chỉ là baseline theo điểm RFM và phải được đối chiếu với cluster profile sau khi K-Means được nghiệm thu."]\n        self._record("Q07",q,"rfm_with_scores.csv",table,figure,"S7-Q07",pts,["Điểm RFM mô tả không phải kết quả K-Means và không được dùng làm tên cụm cuối.","Khuyến nghị marketing cuối cùng bị hoãn đến Giai đoạn 9 sau cluster profiling."],"TV4 + TV3","TV1","RFM profiling readiness; heatmap; customer segmentation preparation")\n        return {"QuestionID":"Q07","OriginalQuestion":old,"DataLimitation":"Stage 7 prohibits K-Means and no cluster assignment file exists.","MinimalAdjustment":q,"GoalPreserved":"Prepare evidence-based profiling and strategy baseline without inventing clusters.","Status":"ADJUSTED_MINIMALLY"}\n\n    def execute_all(self):\n        self.analyze_time_trend(); self.analyze_revenue_contribution(); self.analyze_customer_behavior(); self.analyze_cancellations_returns(); self.analyze_rfm(); self.analyze_monetary_concentration(); change=self.analyze_descriptive_profile_readiness()\n        return pd.DataFrame([change])\n\n    def export_registries(self,change_log,input_verification):\n        results=pd.DataFrame(self.results); figures=pd.DataFrame(self.figures); interpretations=pd.DataFrame(self.interpretations); limitations=pd.DataFrame(self.limitations)\n        acceptance=results[["QuestionID","TechnicalStatus","PeerReviewStatus","TablePath","FigurePath","CrossChecker"]].copy(); acceptance["AcceptanceStatus"]="TECHNICALLY PASS, PEER REVIEW PENDING"\n        rubric=pd.DataFrame([\n            {"Requirement":"At least five questions","Questions":"Q01-Q07","Evidence":"analysis_results_registry.csv","Coverage":"PASS"},\n            {"Requirement":"Descriptive statistics","Questions":"Q03,Q05","Evidence":"q03,q05 tables","Coverage":"PASS"},\n            {"Requirement":"Groupby/pivot","Questions":"Q01,Q02,Q04,Q07","Evidence":"question tables","Coverage":"PASS"},\n            {"Requirement":"Group comparison","Questions":"Q02,Q04,Q07","Evidence":"bar/heatmap tables","Coverage":"PASS"},\n            {"Requirement":"Time trend","Questions":"Q01","Evidence":"line chart and monthly table","Coverage":"PASS"},\n            {"Requirement":"Revenue","Questions":"Q01,Q02,Q06","Evidence":"sterling-labelled tables/figures","Coverage":"PASS"},\n            {"Requirement":"RFM","Questions":"Q05,Q06,Q07","Evidence":"RFM tables/figures","Coverage":"PASS"},\n            {"Requirement":"Seven chart types","Questions":"Q01-Q07","Evidence":"figure_registry.csv","Coverage":"PASS"},\n            {"Requirement":"Interpretation and limitations","Questions":"Q01-Q07","Evidence":"interpretation/limitation registries","Coverage":"PASS"},\n        ])\n        outputs={"analysis_results_registry.csv":results,"figure_registry.csv":figures,"question_acceptance_matrix.csv":acceptance,"interpretation_registry.csv":interpretations,"limitation_registry.csv":limitations,"rubric_coverage_stage7.csv":rubric,"question_change_log.csv":change_log,"input_verification.csv":input_verification}\n        for name,df in outputs.items(): df.to_csv(self.paths.tables/name,index=False,encoding="utf-8-sig")\n        return outputs\n\n    def verify_input_integrity(self):\n        rows=[]\n        for name in EXPECTED:\n            p=self.paths.input/name; self.hash_after[name]=self.sha256(p); rows.append({"File":name,"SHA256Before":self.hash_before[name],"SHA256After":self.hash_after[name],"Unchanged":self.hash_before[name]==self.hash_after[name],"SizeBytes":p.stat().st_size})\n        result=pd.DataFrame(rows); result.to_csv(self.paths.tables/"input_checksum_report.csv",index=False,encoding="utf-8-sig")\n        if not result.Unchanged.all(): raise AnalysisAcceptanceError("Input checksum changed")\n        return result\n'
(PATHS["src"]/"analysis_engine.py").write_text(module_source,encoding="utf-8")
if str(PATHS["src"]) not in sys.path: sys.path.insert(0,str(PATHS["src"]))
analysis_engine=importlib.import_module("analysis_engine"); importlib.reload(analysis_engine)
from analysis_engine import AnalysisEngine,Paths
engine=AnalysisEngine(Paths(PROJECT_ROOT)); input_verification=engine.validate_inputs(); display(input_verification)


In [ ]:
# CELL 8 - Load Analysis Question Registry
question_registry = pd.DataFrame([
{"QuestionID":"Q01","Question":"Số hóa đơn, số giao dịch và tổng TransactionAmount của các giao dịch mua hợp lệ thay đổi theo tháng như thế nào?","Dataset":"online_retail_clean.csv","Columns":"InvoiceDate; InvoiceNo; TransactionAmount","Operation":"monthly groupby, nunique, sum","OutputTable":"q01_monthly_trends.csv","FigureID":"FIG-Q01-01","ChartType":"Line chart","Owner":"TV3","CrossChecker":"TV1","Rubric":"time trend; groupby; revenue"},
{"QuestionID":"Q02","Question":"Doanh thu mua hàng, số hóa đơn và số khách hàng có định danh phân bố theo quốc gia như thế nào, và mức độ tập trung đóng góp ra sao?","Dataset":"online_retail_clean.csv","Columns":"Country; TransactionAmount; InvoiceNo; CustomerID","Operation":"groupby, nunique, share, cumulative share","OutputTable":"q02_country_contribution.csv","FigureID":"FIG-Q02-01","ChartType":"Horizontal bar chart","Owner":"TV3","CrossChecker":"TV2","Rubric":"group comparison; revenue"},
{"QuestionID":"Q03","Question":"Giá trị hóa đơn, số loại sản phẩm trong hóa đơn và số hóa đơn theo khách hàng phân bố như thế nào?","Dataset":"online_retail_rfm_eligible.csv","Columns":"CustomerID; InvoiceNo; StockCode; TransactionAmount","Operation":"invoice/customer aggregation, describe","OutputTable":"q03_customer_behavior.csv","FigureID":"FIG-Q03-01","ChartType":"Histogram + Boxplot","Owner":"TV3","CrossChecker":"TV4","Rubric":"descriptive statistics; behavior"},
{"QuestionID":"Q04","Question":"Các giao dịch hủy, trả hàng và giao dịch không hợp lệ được tách riêng phân bố như thế nào theo loại lý do, thời gian, quốc gia và mã sản phẩm?","Dataset":"online_retail_cancellations_returns.csv","Columns":"flags; InvoiceDate; Country; StockCode","Operation":"exclusive masks, groupby, pivot","OutputTable":"q04_exception_profile.csv","FigureID":"FIG-Q04-01","ChartType":"Stacked bar chart","Owner":"TV2","CrossChecker":"TV1","Rubric":"data quality; group comparison"},
{"QuestionID":"Q05","Question":"Recency, Frequency và Monetary phân bố, lệch và liên hệ với nhau như thế nào sau khi bảng RFM được tạo bằng giao dịch đủ điều kiện?","Dataset":"rfm_customers.csv","Columns":"Recency; Frequency; Monetary","Operation":"describe, skewness, Spearman correlation","OutputTable":"q05_rfm_diagnostics.csv","FigureID":"FIG-Q05-01","ChartType":"Heatmap","Owner":"TV3","CrossChecker":"TV4","Rubric":"RFM; correlation"},
{"QuestionID":"Q06","Question":"Monetary có tập trung vào một nhóm nhỏ khách hàng hay không, và nhóm đóng góp Monetary cao có đặc điểm Recency và Frequency như thế nào?","Dataset":"rfm_customers.csv","Columns":"CustomerID; Recency; Frequency; Monetary","Operation":"rank, share, cumulative share","OutputTable":"q06_monetary_concentration.csv","FigureID":"FIG-Q06-01","ChartType":"Pareto chart","Owner":"TV3","CrossChecker":"TV4","Rubric":"RFM; revenue concentration"},
{"QuestionID":"Q07","Question":"Baseline profiling trước clustering theo tổ hợp điểm RFM","Dataset":"rfm_with_scores.csv","Columns":"R/F/M scores and continuous RFM","Operation":"score-combination profile","OutputTable":"q07_rfm_score_profile_baseline.csv","FigureID":"FIG-Q07-01","ChartType":"Profile heatmap","Owner":"TV4 + TV3","CrossChecker":"TV1","Rubric":"segmentation preparation"}])
display(question_registry)


In [ ]:
# CELL 9 - Execute Question 1
engine.analyze_time_trend()
print("Q01 executed")


In [ ]:
# CELL 10 - Execute Question 2
engine.analyze_revenue_contribution()
print("Q02 executed")


In [ ]:
# CELL 11 - Execute Question 3
engine.analyze_customer_behavior()
print("Q03 executed")


In [ ]:
# CELL 12 - Execute Question 4
engine.analyze_cancellations_returns()
print("Q04 executed")


In [ ]:
# CELL 13 - Execute Question 5
engine.analyze_rfm()
print("Q05 executed")


In [ ]:
# CELL 14 - Execute Question 6
engine.analyze_monetary_concentration()
print("Q06 executed")


In [ ]:
# CELL 15 - Execute Question 7
question_change=engine.analyze_descriptive_profile_readiness()
question_change_log=pd.DataFrame([question_change]); display(question_change_log)


In [ ]:
# CELL 16 - Build Analysis Results Registry
registries=engine.export_registries(question_change_log,input_verification)
analysis_results_registry=registries["analysis_results_registry.csv"]
display(analysis_results_registry)


In [ ]:
# CELL 17 - Build Figure Registry
figure_registry=registries["figure_registry.csv"]
display(figure_registry)


In [ ]:
# CELL 18 - Interpretation and Limitation Registries
display(registries["interpretation_registry.csv"]); display(registries["limitation_registry.csv"])


In [ ]:
# CELL 19 - Stage 7 Rubric Coverage
rubric=registries["rubric_coverage_stage7.csv"]; display(rubric)
if not rubric.Coverage.eq("PASS").all(): raise AssertionError("Rubric coverage failed")


In [ ]:
# CELL 20 - Output Read-Back Verification
paths=[]
for p in sorted(list(PATHS["tables"].glob("*.csv"))):
 try:
  df=pd.read_csv(p,low_memory=False); ok=not df.empty
 except Exception as e: ok=False
 paths.append({"Path":str(p.relative_to(PROJECT_ROOT)),"Exists":p.exists(),"SizeBytes":p.stat().st_size,"Readable":ok,"Rows":len(df) if ok else None,"Status":"PASS" if ok else "FAIL"})
output_verification=pd.DataFrame(paths); output_verification.to_csv(PATHS["tables"]/"output_verification.csv",index=False,encoding="utf-8-sig"); display(output_verification)
if not output_verification.Status.eq("PASS").all(): raise AssertionError("Read-back failed")


In [ ]:
# CELL 21 - Chart File Validation
from PIL import Image
chart=[]
for _,x in figure_registry.iterrows():
 p=PROJECT_ROOT/x.Path
 try:
  with Image.open(p) as im: w,h=im.size
  ok=p.stat().st_size>0 and w>=400 and h>=300 and bool(str(x.Title).strip()) and bool(str(x.XLabel).strip()) and bool(str(x.YLabel).strip())
  err=""
 except Exception as e: ok=False; w=h=0; err=str(e)
 chart.append({"FigureID":x.FigureID,"Path":x.Path,"SizeBytes":p.stat().st_size if p.exists() else 0,"Width":w,"Height":h,"TitleValid":bool(str(x.Title).strip()),"XLabelValid":bool(str(x.XLabel).strip()),"YLabelValid":bool(str(x.YLabel).strip()),"No3D":True,"Status":"PASS" if ok else "FAIL","Error":err})
chart_validation=pd.DataFrame(chart); chart_validation.to_csv(PATHS["tables"]/"chart_validation.csv",index=False,encoding="utf-8-sig"); display(chart_validation)
if not chart_validation.Status.eq("PASS").all(): raise AssertionError("Chart validation failed")


In [ ]:
# CELL 22 - Automated Acceptance Tests
import os,subprocess
(PATHS["tests"]/"test_stage7_acceptance.py").write_text('"""Acceptance tests for externally executed Stage 7 EDA outputs."""\nfrom pathlib import Path\nimport re\nimport pandas as pd\nfrom PIL import Image\n\nROOT=Path(__file__).resolve().parents[1]\nT=ROOT/"outputs"/"tables"/"analysis"; F=ROOT/"outputs"/"figures"/"analysis"\n\ndef test_seven_questions_and_outputs():\n    r=pd.read_csv(T/"analysis_results_registry.csv")\n    assert len(r)==7 and r.QuestionID.nunique()==7 and set(r.QuestionID)=={f"Q{i:02d}" for i in range(1,8)}\n    for _,x in r.iterrows():\n        assert (ROOT/x.TablePath).exists() and (ROOT/x.TablePath).stat().st_size>0\n        assert (ROOT/x.FigurePath).exists() and (ROOT/x.FigurePath).stat().st_size>0\n        assert not pd.read_csv(ROOT/x.TablePath,low_memory=False).empty\n\ndef test_seven_distinct_chart_types_and_unique_figures():\n    f=pd.read_csv(T/"figure_registry.csv")\n    assert f.FigureID.is_unique and f.Path.is_unique and len(f)==7\n    normalized=set()\n    for value in f.ChartType:\n        for item in re.split(r"\\s*\\+\\s*|\\s*;\\s*", value.lower()):\n            item=item.strip()\n            if item == "profile heatmap": item = "heatmap"\n            normalized.add(item)\n    assert {"line chart","horizontal bar chart","histogram","boxplot","stacked bar chart","heatmap","pareto chart"}.issubset(normalized)\n    assert not f.ChartType.str.contains("3d",case=False).any()\n\ndef test_interpretations_limitations_and_source_links():\n    i=pd.read_csv(T/"interpretation_registry.csv"); l=pd.read_csv(T/"limitation_registry.csv")\n    assert set(i.QuestionID)=={f"Q{x:02d}" for x in range(1,8)}\n    assert set(l.QuestionID)==set(i.QuestionID)\n    assert i.Text.str.len().gt(10).all() and l.Text.str.len().gt(10).all()\n    assert i.SourceTable.map(lambda p:(ROOT/p).exists()).all()\n    forbidden=re.compile(r"\\b(causes?|leads? to|results? in)\\b",re.I)\n    assert not i.Text.map(lambda x:bool(forbidden.search(str(x)))).any()\n\ndef test_chart_files_valid():\n    f=pd.read_csv(T/"figure_registry.csv")\n    for _,x in f.iterrows():\n        path=ROOT/x.Path\n        assert path.exists() and path.stat().st_size>0\n        with Image.open(path) as im:\n            assert im.width>=400 and im.height>=300\n        assert str(x.Title).strip() and str(x.XLabel).strip() and str(x.YLabel).strip()\n\ndef test_input_integrity_and_output_readback():\n    c=pd.read_csv(T/"input_checksum_report.csv"); assert c.Unchanged.all()\n    o=pd.read_csv(T/"output_verification.csv"); assert o.Status.eq("PASS").all()\n\ndef test_rubric_coverage_and_acceptance():\n    rubric=pd.read_csv(T/"rubric_coverage_stage7.csv"); assert rubric.Coverage.eq("PASS").all()\n    matrix=pd.read_csv(T/"stage7_acceptance_matrix.csv"); assert matrix.Status.eq("PASS").all()\n\ndef test_no_clustering_artifacts():\n    files=[p.name.lower() for p in ROOT.rglob("*") if p.is_file()]\n    assert not any("pca" in x or "kmeans" in x or "cluster_id" in x for x in files)\n',encoding="utf-8")
stage7_acceptance=pd.DataFrame([{"CheckID":"S7-01","Condition":"Seven questions executed","Status":"PASS" if len(analysis_results_registry)==7 else "FAIL"},{"CheckID":"S7-02","Condition":"All outputs read back","Status":"PASS" if output_verification.Status.eq("PASS").all() else "FAIL"},{"CheckID":"S7-03","Condition":"Charts validate","Status":"PASS" if chart_validation.Status.eq("PASS").all() else "FAIL"},{"CheckID":"S7-04","Condition":"Rubric coverage","Status":"PASS" if rubric.Coverage.eq("PASS").all() else "FAIL"},{"CheckID":"S7-05","Condition":"No peer review fabricated","Status":"PASS" if analysis_results_registry.PeerReviewStatus.eq("PENDING").all() else "FAIL"}])
stage7_acceptance.to_csv(PATHS["tables"]/"stage7_acceptance_matrix.csv",index=False,encoding="utf-8-sig")
env=os.environ.copy(); env["PYTHONPATH"]=str(PATHS["src"])
r=subprocess.run([sys.executable,"-m","pytest","-q",str(PATHS["tests"]/"test_stage7_acceptance.py")],cwd=PROJECT_ROOT,env=env,text=True,capture_output=True)
pytest_text=r.stdout+"\n"+r.stderr; (PATHS["logs"]/"pytest_output.txt").write_text(pytest_text,encoding="utf-8"); print(pytest_text)
if r.returncode!=0: raise AssertionError("Pytest failed")


In [ ]:
# CELL 23 - Input SHA-256 After Execution
checksum_report=engine.verify_input_integrity(); display(checksum_report)
(PATHS["logs"]/"input_checksum_report.txt").write_text(checksum_report.to_string(index=False),encoding="utf-8")


In [ ]:
# CELL 24 - Final Machine-Readable Acceptance Summary
from datetime import datetime,timezone
import platform
question_count=len(analysis_results_registry); tech_pass=int(analysis_results_registry.TechnicalStatus.str.startswith("TECHNICALLY PASS").sum()); peer_reviewed=int(analysis_results_registry.PeerReviewStatus.eq("REVIEWED").sum())
atomic={"line chart","horizontal bar chart","histogram","boxplot","stacked bar chart","heatmap","pareto chart"}
created=set()
for value in figure_registry.ChartType.str.lower():
 for item in re.split(r"\s*\+\s*|\s*;\s*",value):
  if item=="profile heatmap": item="heatmap"
  created.add(item)
failed=[]
if question_count!=7 or tech_pass!=7: failed.append("QUESTIONS")
if not atomic.issubset(created): failed.append("CHART_TYPES")
if r.returncode!=0: failed.append("PYTEST")
if not checksum_report.Unchanged.all(): failed.append("INPUT_INTEGRITY")
candidate="PASS" if not failed else "FAIL"
summary="\n".join([f"STAGE_7_EXECUTION_STATUS = {candidate}",f"STAGE_7_ACCEPTANCE_CANDIDATE = {candidate}","","OFFICIAL_QUESTIONS = 7",f"QUESTIONS_EXECUTED = {question_count}/7",f"QUESTIONS_TECHNICALLY_PASSED = {tech_pass}/7",f"QUESTIONS_PEER_REVIEWED = {peer_reviewed}/7 (PENDING)",f"RESULT_TABLES_CREATED = {question_count}",f"FIGURES_CREATED = {len(figure_registry)}",f"DISTINCT_CHART_TYPES = {len(atomic.intersection(created))}","",f"INPUT_FILES_VERIFIED = {len(input_verification)}/5",f"INPUT_FILES_UNCHANGED = {'YES' if checksum_report.Unchanged.all() else 'NO'}",f"OUTPUT_READ_BACK = {'PASS' if output_verification.Status.eq('PASS').all() else 'FAIL'}",f"CHART_VALIDATION = {'PASS' if chart_validation.Status.eq('PASS').all() else 'FAIL'}",f"RUBRIC_COVERAGE = {'PASS' if rubric.Coverage.eq('PASS').all() else 'FAIL'}",f"PYTEST_STATUS = {'PASS' if r.returncode==0 else 'FAIL'}","",f"ANALYSIS_RESULTS_CREATED = {'YES' if candidate=='PASS' else 'NO'}",f"EDA_EXECUTED = {'YES' if candidate=='PASS' else 'NO'}","KMEANS_EXECUTED = NO","K_SELECTED = NO","CLUSTER_NAMES_CREATED = NO","PCA_CREATED = NO","REPORT_WRITTEN = NO","SLIDES_CREATED = NO","VIVA_CONTENT_CREATED = NO","",'OPEN_CONDITIONS = ["Final Chat evidence review pending", "Peer review evidence may remain pending", "SQLite second-format confirmation remains open"]',f"FAILED_CHECKS = {failed if failed else 'NONE'}",'NOT_VERIFIED_ITEMS = ["Peer review confirmations"]'])
(PATHS["logs"]/"acceptance_summary.txt").write_text(summary,encoding="utf-8"); (PATHS["logs"]/"execution_log.txt").write_text(f"ENTRY_POINT=AnalysisEngine execution\nSTATUS={candidate}\nFINISHED_UTC={datetime.now(timezone.utc).isoformat()}\nEXCEPTION=NONE\n",encoding="utf-8"); (PATHS["logs"]/"environment_info.txt").write_text(f"Python={sys.version}\nPlatform={platform.platform()}\nProjectRoot={PROJECT_ROOT}\n",encoding="utf-8"); print(summary)
if candidate!="PASS": raise AssertionError("Acceptance candidate failed")


In [ ]:
# CELL 25 - Build and Download Evidence ZIP
import zipfile
from google.colab import files
evidence=[p for p in PROJECT_ROOT.rglob("*") if p.is_file() and p.name not in {"online_retail_clean.csv","online_retail_rfm_eligible.csv","online_retail_cancellations_returns.csv"}]
# exclude uploaded inputs and include code, tests, tables, figures and logs
zip_path=PROJECT_ROOT/"DT25_Stage7_EDA_Evidence.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
 for p in evidence:
  if PATHS["input"] not in p.parents: z.write(p,p.relative_to(PROJECT_ROOT))
print(zip_path,zip_path.stat().st_size); files.download(str(zip_path))
